In [1]:
import pandas as pd
import configparser as cp
import psycopg2
from openpyxl.utils import column_index_from_string
from openpyxl.utils import get_column_letter

In [2]:
config = cp.RawConfigParser()
config_files = ['584/nss76_593_Table_1.0.properties']


print(config_files)

['584/nss76_593_Table_1.0.properties']


In [3]:
#read property file

for file in config_files:
    print('Currently Read File  :- ' + file)
    config.read(file)

    
    ip = config.get('database_details', 'database.ip')
    port = config.get('database_details', 'database.port')
    username = config.get('database_details', 'database.username')
    password = config.get('database_details', 'database.password')
    dbname = config.get('database_details', 'database.dbname')


    
    connection = psycopg2.connect(database=dbname, user=username, password=password, host=ip, port=port)
    cursor = connection.cursor()
    
    sheet_location = config.get('master_properties', 'sheet_path')
    print('reading excel file:',sheet_location)
    
    
    
    #read property env.tables.unique.sheets and start the nested loops
    unique_sheet_count = int(config.get('nss76_tables_for_etl', 'nss76.tables.unique.sheets'))
    
    #below loop for unique sheets only
    for i in range(unique_sheet_count):
        print('Starting for i(Unique Sheets Count):',i)
        #run below loop for blocks that are split across sheets for "each" unique sheet
        block_count = int(config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.block.count'))
        header_names = config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.header.names').split(',')
        row_seggregation_level = int(config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.rows.seggregation.level.count'))
        col_seggregation_level = int(config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.cols.seggregation.level.count'))
        row_start = int(config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.row.start'))
        row_end = int(config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.row.end'))
        col_start = column_index_from_string(config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.col.start'))
        col_end = column_index_from_string(config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.col.end'))
    
        row_seggregation_values_list = []
        row_seggregation_criteria_list = []

        col_seggregation_foreign_list = []
        col_seggregation_loopValue_list = []
        col_seggregation_repeteAfter_list = []
        col_seggregation_foreignValue_list = []
        
        for k in range(int(row_seggregation_level)):
            row_seggregation_values_list.append(((config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.rows.seggregation.'+str(k+1)+'.values')).split(',')))
            row_seggregation_criteria_list.append(((config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.rows.seggregation.'+str(k+1)+'.criteria'))))
        
        
        for l in range(int(col_seggregation_level)):
            col_seggregation_foreign_list.append(config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.cols.seggregation.'+str(l+1)+'.foreign'))
            col_seggregation_loopValue_list.append(config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.cols.seggregation.'+str(l+1)+'.loopValue'))
            col_seggregation_repeteAfter_list.append(config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.cols.seggregation.'+str(l+1)+'.totalRows'))
            col_seggregation_foreignValue_list.append(config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.cols.seggregation.'+str(l+1)+'.foreignValue'))

        #print('row_seggregation_criteria_list:',row_seggregation_criteria_list)
        #print('row_seggregation_values_list:',row_seggregation_values_list)
        
        #print('col_seggregation_criteria_list:',col_seggregation_criteria_list)
        #print('col_seggregation_values_list:',col_seggregation_values_list)
    
        
        for j in range(block_count):
            header_values = config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.block.'+str(j+1)+'.sheet.header.values').split(',')
            sheet_name = config.get('nss76_tables_for_etl', 'nss76.tables.table.'+str(i+1)+'.block.'+str(j+1)+'.sheet')
            input_df = pd.read_excel(sheet_location,sheet_name)
            print('Read Excel Sheet:',sheet_name)
            for row in range(row_start-2,row_end-1):
                for col in range(col_start-1,col_end):
                    insert_query_prefix = "insert into nss76_fact("+""
                    insert_query_suffix = " values("+"'"

                    
                    for o in range(len(header_names)):
                        insert_query_prefix = insert_query_prefix + header_names[o] +","
                        insert_query_suffix = insert_query_suffix + header_values[o]+"','"

                    col_seggregation_foreignValue_list = [
                        x.split(",") if isinstance(x, str) else x
                        for x in col_seggregation_foreignValue_list
                    ]

                    
                    for m in range(len(col_seggregation_foreign_list)):

                        insert_query_prefix += col_seggregation_foreign_list[m] + ","

                        repeat = int(col_seggregation_repeteAfter_list[m])
                        loop = int(col_seggregation_loopValue_list[m])
                        base = (row + 2) - 6

                        if m == 0:
                            new_val = base // repeat

                        elif m == 1:
                            new_val_1 = (base // repeat)
                            new_val =  (base //  20) - (39 * new_val_1)

                        elif m == 2:
                            new_val = base % loop

                        # ✅ Safe indexing
                        values_list = col_seggregation_foreignValue_list[m]

                        if new_val < len(values_list):
                            insert_query_suffix += values_list[new_val] + "','"
                        else:
                            insert_query_suffix += "NULL','"   # safety
                    
                                     
                    
                    indicator_val = input_df.iloc[row].values[col]
                   
                    # print('Inserting, indicator_val['+str(row+2)+']['+str(get_column_letter(col+1))+']:',indicator_val)
    
    
                    insert_query_prefix = insert_query_prefix  +"value,"
                    # rounded_number = round(float(indicator_val), 2)
                    insert_query_suffix = insert_query_suffix + str(indicator_val)+"','"
    
                    insert_query_prefix = insert_query_prefix  +"created_by,"
                    insert_query_suffix = insert_query_suffix + 'system' +"','"
                    insert_query_prefix = insert_query_prefix  +"updated_by,"
                    insert_query_suffix = insert_query_suffix + ' system' +"','"
    
                    from datetime import datetime

                    # insert_query_prefix = insert_query_prefix  +"year,"
                    # insert_query_suffix = insert_query_suffix + sheet_year +"','"
                    timestamp = datetime.now().strftime("%Y%m%d%H%M%S%f")
    
                    nfhs_fact_code = f"nss76_{sheet_name}_{row+2}_{get_column_letter(col+1)}_{indicator_val}_{timestamp}"
                    nfhs_fact_code = nfhs_fact_code.replace(' ', '')
                    insert_query_prefix = insert_query_prefix + "nss76_fact_code)"
                    insert_query_suffix = insert_query_suffix + nfhs_fact_code+"')"
    
                    final_query = insert_query_prefix+insert_query_suffix
                    print(final_query)
                    cursor.execute(final_query)
                     
                    connection.commit()

connection.close()
print('Data insert successfully')



Currently Read File  :- 584/nss76_593_Table_1.0.properties
reading excel file: D:\GIT\nss76\data\Round - 76 Report - 584\Table 10.xlsx
Starting for i(Unique Sheets Count): 0
Read Excel Sheet: Table 10
insert into nss76_fact(indicator_code,sub_indicator_code,sector_code,state_code,principal_source_of_drinking_water_code,value,created_by,updated_by,nss76_fact_code) values('1','1','1','1','1','2.1','system',' system','nss76_Table10_6_D_2.1_20260401115724190580')
insert into nss76_fact(indicator_code,sub_indicator_code,sector_code,state_code,principal_source_of_drinking_water_code,value,created_by,updated_by,nss76_fact_code) values('1','1','1','1','2','4.7','system',' system','nss76_Table10_7_D_4.7_20260401115724207605')
insert into nss76_fact(indicator_code,sub_indicator_code,sector_code,state_code,principal_source_of_drinking_water_code,value,created_by,updated_by,nss76_fact_code) values('1','1','1','1','3','3.9','system',' system','nss76_Table10_8_D_3.9_20260401115724217609')
insert int